# Simple data center: constant-load sizing
Keep the HPP fixed and jointly optimize constant facility load and hourly dispatch.
The objective is revenue from delivered energy at EUR 70/MWh; the YAML reliability_target
is enforced as a minimum hourly full-load factor. EENS is reported afterwards.

Select the hydesign_clean kernel and run all cells. This notebook runs independently;
there is no need to run [simple_datacenter.ipynb](simple_datacenter.ipynb) first.
Both notebooks use the same YAML and model source files.

The EMS uses one CPLEX dispatch solve. Set battery_power_fluctuation_penalty
in YAML (default 0.0; optional trials 0.0001 or 0.001).
Its units are MWh_IT per MW of adjacent change in AC net battery power
(discharge - charge); there is no end-to-start ramp or throughput penalty.
Fixed/basic dispatch maximizes served IT MWh minus this energy-equivalent penalty.
Economic sizing multiplies the penalty by the IT revenue rate to obtain EUR.
Economic reports exclude the artificial penalty; solver objectives/bounds include it.

The unchanged validator rejects simultaneous charging and discharging. The existing
physical formulation permits this, and neither zero nor a positive ramp penalty
guarantees its elimination. The annual reference evaluation can therefore stop with
"Simultaneous battery charging/discharging"; do not use rejected dispatch as a result.


In [ ]:
import os
os.environ['OPENMDAO_REPORTS'] = '0'
import time
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hydesign

repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'hydesign/ems/ems_simple_datacenter.py').is_file()), None)
if repo is None or Path(hydesign.__file__).resolve().parent != (repo / 'hydesign').resolve():
    raise RuntimeError('Select the kernel with this HyDesign checkout installed editable.')
print('Python:', sys.executable)
print('HyDesign:', hydesign.__file__)

from hydesign.assembly.hpp_assembly_simple_datacenter import hpp_model_simple_datacenter as hpp_model
from hydesign.examples import examples_filepath
from hydesign.sizing_simple_datacenter import size_datacenter


## Site and sizing assumptions
The minimum reliability, load bounds and solver limits come from the shared YAML.
The 16 MW value below is used only by the existing assembly during input preparation.


In [ ]:
examples_sites = pd.read_csv(f'{examples_filepath}examples_sites.csv', index_col=0, sep=';')
name = 'Denmark_good_solar'
ex_site = examples_sites.loc[examples_sites.name == name].iloc[0]
latitude = ex_site.latitude
longitude = ex_site.longitude
altitude = ex_site.altitude
input_ts_fn = examples_filepath + ex_site.input_ts_fn
sim_pars_fn = examples_filepath + 'thesis_simple_datacenter/simple_datacenter.yml'

facility_load_mw = 16.0
PUE = 1.15
PPA = 70.0 # Reporting only; unused by dispatch/sizing objectives.
hpp = hpp_model(
    latitude=latitude,
    longitude=longitude,
    altitude=altitude,
    work_dir='./',
    sim_pars_fn=sim_pars_fn,
    input_ts_fn=input_ts_fn,
    facility_load_mw=facility_load_mw,
    pue=PUE,
    ppa_price=PPA,
)
reliability_target = hpp.sim_pars['reliability_target']
print(f'Minimum full-load factor for sizing: {reliability_target:.2%}')
print('Site:', name)
print('Weather:', input_ts_fn)
print('Preparation load [MW] (not the sized result):', facility_load_mw)
print('Hourly weather:', hpp.weather.index[0], 'to', hpp.weather.index[-1], 'N_time =', hpp.N_time)


## Fixed HPP and generation inputs
Wind is 100 MW, PV is 180 MW AC, and the battery is 25 MW / 200 MWh nominal.
These capacities remain fixed during load sizing.

The existing hpp.evaluate(*x) call prepares hourly wind and PV generation. It also
runs an unconstrained fixed-load dispatch because that is part of the existing assembly.
That dispatch is not the sizing result, and its battery schedule is not passed to sizing.
Only the generation profiles and fixed plant parameters are reused; the next solve
chooses its own load and dispatch with the YAML reliability target enforced.


In [ ]:
start = time.time()
clearance = 35
sp = 300
p_rated = 5
Nwt = 20
wind_MW_per_km2 = 7
solar_MW = 180
surface_tilt = 42
surface_azimuth = 180
solar_DCAC = 1.25
b_P = 25
b_E_h = 8

x = [clearance, sp, p_rated, Nwt, wind_MW_per_km2,
     solar_MW, surface_tilt, surface_azimuth, solar_DCAC,
     b_P, b_E_h]
# Prepare generation using the existing fixed-design assembly.
# Disable reliability only for this preparation; keep the YAML target for sizing.
hpp.prob.set_val('reliability_target', 0.0)
hpp.evaluate(*x)
print('Generation preparation time [min]:', (time.time() - start) / 60)
print('Hourly generation prepared:', len(hpp.prob.get_val('wind_t')))


## Size contracted data-center load; keep the HPP fixed
The sizing module creates one constant facility-load variable (1-50 MW). It reuses
the EMS constraints and jointly selects load and dispatch using CPLEX. Wind, PV
and battery ratings are unchanged. The YAML reliability target determines the allowed failed hours.

PPA revenue is reported as EUR 70/MWh times delivered facility energy, including
partial-supply hours, but does not enter the sizing objective. EENS is calculated only afterwards. No data-center construction/operating costs
or shortage penalties are included, so this is served-energy sizing, not yet
an integrated economic optimum for the physical data center.

One solve maximizes served IT energy minus the optional battery fluctuation penalty. CPLEX may certify optimality within the configured 0.01% MIP gap;
the notebook prints that gap. At the time limit, a feasible candidate is returned
with its remaining penalized-objective gap and labelled NOT proven optimal. A solve without a feasible
solution still raises an error.


In [ ]:
sim_pars = hpp.sim_pars
result = size_datacenter(
    wind_ts=hpp.prob.get_val('wind_t'),
    solar_ts=hpp.prob.get_val('solar_t'),
    b_P=b_P,
    b_E=b_P * b_E_h,
    battery_depth_of_discharge=sim_pars['battery_depth_of_discharge'],
    battery_charge_efficiency=sim_pars['battery_charge_efficiency'],
    battery_initial_soc=sim_pars['battery_initial_soc'],
    pue=PUE,
    reliability_target=reliability_target,
    battery_power_fluctuation_penalty=sim_pars.get('battery_power_fluctuation_penalty', 0.0),
    ppa_price=PPA,
    facility_load_min_mw=sim_pars['facility_load_min_mw'],
    facility_load_max_mw=sim_pars['facility_load_max_mw'],
    epsilon_mw=sim_pars['epsilon_mw'],
    time_limit_seconds=sim_pars['sizing_time_limit_seconds'],
    threads=sim_pars['ems_threads'],
    mip_relative_gap=sim_pars['sizing_mip_relative_gap'],
    allow_time_limit_incumbent=sim_pars['sizing_allow_time_limit_incumbent'],
)


## Sized load and hourly results
All metrics below describe the load and dispatch selected by the sizing optimization.


In [ ]:
print('Solver:', result['solver_status'])
if result['solver_status']['optimal']:
    print('Optimal within the configured CPLEX tolerance.')
else:
    print('FEASIBLE CANDIDATE - NOT PROVEN OPTIMAL')
primary = result['solver_status']['stages'][0]
print('Objective units:', primary['objective_units'])
print('Unpenalized physical/economic objective:', primary['unpenalized_objective'])
print('Net battery-power fluctuation [MW]:', primary['battery_power_fluctuation_mw'])
print('Artificial penalty:', primary.get('battery_penalty_eur', primary.get('battery_penalty_equivalent_it_mwh')))
print(f"Remaining penalized served-IT-energy gap: {primary.get('relative_gap', 0.0):.2%}")
print(f"Penalized objective upper bound: {primary.get('best_bound', primary['objective']):,.6f} MWh_IT")
print(f"Contracted facility load: {result['facility_load_mw']:.6f} MW")
print(f"Corresponding IT load: {result['firm_demand_it_mw']:.6f} MW")
print('Load bounds [MW]:', result['facility_load_bounds_mw'])
print('Necessary physical upper bound [MW]:', result['physical_load_upper_mw'])
print('Upper/lower bound active:', result['upper_bound_active'], result['lower_bound_active'])
if result['upper_bound_active']:
    print('WARNING: increase the upper load bound before interpreting this as the unrestricted optimum.')

metrics = result['metrics']
print(f"Full-load factor: {metrics['full_load_factor']:.4%} (minimum {reliability_target:.2%})")
print(f"Failed hours: {metrics['LOLH']:.0f} / {metrics['total_hours']:.0f}")
print(f"Delivered facility energy: {metrics['delivered_energy_mwh']:.3f} MWh/year")
print(f"PPA revenue: EUR {metrics['ppa_revenue_eur']:,.2f}/year")
print(f"EENS (reported only): {metrics['EENS']:.3f} MWh/year")
print(f"Longest chronological shortage: {metrics['longest_shortage_hours']:.0f} h")
print(f"Minimum supplied facility power: {metrics['minimum_supplied_mw']:.3f} MW")
print(f"Maximum shortfall: {metrics['maximum_shortfall_mw']:.3f} MW")
allowed_failed_hours = int(np.floor(
    (1 - reliability_target) * metrics['total_hours'] + 1e-9
))
assert metrics['LOLH'] <= allowed_failed_hours
assert metrics['full_load_factor'] >= reliability_target - 1e-12
print('Physical validation passed:', result['validation'])

# The sizing solution is in result; hpp.prob contains only the preparation run.
# All results and plots below use the sizing solution.
hourly = pd.DataFrame(result['hourly'], index=hpp.weather.index)
soc_index = hpp.weather.index.append(pd.DatetimeIndex([hpp.weather.index[-1] + pd.Timedelta(hours=1)]))
soc_mwh = pd.Series(result['soc_mwh'], index=soc_index, name='soc_mwh')
P_supplied = hourly.served_facility_mw.to_numpy()
display(hourly.head())


## Sized-load dispatch plots
These plots use result['hourly'] and result['soc_mwh'] from the sizing solve.


In [ ]:
# Plot a week around the largest shortfall, or the first week if none occurs.
peak = int(np.argmax(hourly.unserved_facility_mw.to_numpy()))
start = max(0, min(peak - 72, len(hourly) - 168))
stop = min(start + 168, len(hourly))
window = hourly.iloc[start:stop]
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
axes[0].plot(window.index, window.renewable_available_mw, label='Available wind + PV')
axes[0].plot(window.index, window.served_facility_mw, label='Served facility load')
axes[0].plot(window.index, window.facility_load_mw, '--', label='Requested facility load')
axes[0].plot(window.index, window.unserved_facility_mw, label='Unserved facility load')
axes[0].set_ylabel('MW'); axes[0].legend()
axes[1].plot(window.index, window.discharge_mw - window.charge_mw, label='Battery: + discharge / - charge')
axes[1].axhline(0, color='grey', linewidth=0.5)
axes[1].set_ylabel('MW'); axes[1].legend()
axes[2].plot(soc_mwh.iloc[start:stop+1], label='Stored energy')
axes[2].axhline((1-hpp.sim_pars['battery_depth_of_discharge'])*b_P*b_E_h, linestyle='--', color='grey')
axes[2].axhline(b_P*b_E_h, linestyle='--', color='grey')
axes[2].set_ylabel('MWh'); axes[2].legend()
fig.autofmt_xdate(); plt.tight_layout(); plt.show()


## Formulation and limits
The existing HyDesign class / ComponentWrapper / assembly structure is retained.
The load-sizing variable lives in sizing_simple_datacenter.py; shared hourly
constraints live in ems_simple_datacenter.py. HPP sizing is not implemented.

For every hour, served IT + unserved IT = facility load / PUE. Renewable power +
battery discharge = served facility power + battery charge + curtailment.
SOC has T+1 states, all transitions connected, and equal 50% initial/terminal states.
During load sizing, a binary flag permits shortfall in an hour; no more than floor((1-R)*T) flags may
be one. Unflagged hours must serve the load fully. The reported failure count uses
unserved facility power > 0.001 MW, independently of the binary flags.

At a fixed load, maximizing paid delivered energy is mathematically equivalent to
minimizing shortfall; with load sizing it is a different optimization. The explicit
hourly constraint in load sizing uses the YAML reliability target; the input-preparation dispatch has no minimum reliability constraint. EENS has no objective coefficient.
Different served-energy-optimal dispatches may still have different shortage patterns.
No additional limit is imposed on the length or depth of shortages within the allowed failed hours.

This is one supplied 8760-hour weather year with perfect foresight, not a multi-year
reliability guarantee. The 2012 dataset ends on December 30. N_life=1 supports
hpp_base input handling; no lifetime dispatch is run. Its weather Price column
is unused; PPA revenue is reporting only and has no effect on dispatch.

Conceptual reference: [Thijs EMS](https://github.com/thijsvanwaveren/hydesign/blob/a51e60652a1f55a2b055d3b07352a4ed541898c9/hydesign/ems/ems_offgrid_datacenter.py).
The final-hour SOC defect is corrected and no claim is made to reproduce 96.96%.
